# PyTorch: derivatives as functions

Section 15.2 of the notes.

`torch.func` turns differentiation into a transformation of functions, in the
notation of Chapter 3. It keeps no state, so it suits derivatives that appear
inside a loss, and per-sample derivatives.

In [ ]:
import timeit

import torch
from torch.func import grad, vmap, jacrev, jacfwd, hessian

torch.set_default_dtype(torch.float64)


def f(x):
    return torch.sin(x**2)


x = torch.tensor(1.0)
print("f'(1) ", grad(f)(x).item())
print("f''(1)", grad(grad(f))(x).item())

## 1. A batch without a loop

`vmap` applies a function to every entry of a batch. The derivative of each
entry is computed independently, which is what the last notebook did with a
loop over the batch.

In [ ]:
xs = torch.linspace(0, 2, 5)
print("vmap(grad(f)):", vmap(grad(f))(xs))
print("by hand:      ", 2 * xs * torch.cos(xs**2))

## 2. Jacobians, forward and reverse

`jacrev` uses one reverse sweep per output, `jacfwd` one forward sweep per
input. Section 3.4 says which is cheaper, and the timing below shows it on a
map with many inputs and one output.

In [ ]:
def g(x):
    return torch.stack([x[0] * x[1], torch.sin(x[0] + x[1])])


point = torch.tensor([0.5, 2.0])
print(jacrev(g)(point))
print(torch.allclose(jacrev(g)(point), jacfwd(g)(point)))

In [ ]:
n = 256
A = torch.randn(1, n) / n**0.5


def h(x):
    return torch.tanh(A @ x).squeeze()


z = torch.randn(n)
t_rev = min(timeit.repeat(lambda: jacrev(h)(z), repeat=3, number=20)) / 20
t_fwd = min(timeit.repeat(lambda: jacfwd(h)(z), repeat=3, number=20)) / 20
print(
    f"one output, {n} inputs:  jacrev {1e6 * t_rev:7.1f} us   jacfwd {1e6 * t_fwd:7.1f} us"
)

## 3. Per-sample gradients

The gradient of the loss of *each* sample, rather than of their sum, is a
quantity that the stateful interface cannot produce without a loop over the
samples.

In [ ]:
import torch.nn as nn
from torch.func import functional_call

net = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
params = {k: v.detach() for k, v in net.named_parameters()}


def loss_one(params, x, y):
    pred = functional_call(net, params, (x.unsqueeze(0),))
    return ((pred.squeeze() - y) ** 2).squeeze()


X = torch.randn(4, 2)
Y = torch.randn(4)

per_sample = vmap(grad(loss_one), in_dims=(None, 0, 0))(params, X, Y)
print({k: tuple(v.shape) for k, v in per_sample.items()})

## 4. The Laplacian of a network

For the residual of Chapter 5 only the diagonal of the Hessian is needed. The
full Hessian costs more, and for a scalar field in one dimension the two agree.

In [ ]:
def u(x):
    return torch.sin(x[0] ** 2) * torch.exp(x[1])


point = torch.tensor([0.7, -0.3])
H = hessian(u)(point)
print("Hessian:\n", H)
print("Laplacian:", torch.diagonal(H).sum().item())